# TinyGPT
## GPT-2 (124M) from scratch, fine-tuned on Shakespeare + Math

**Run on Kaggle with GPU P100 enabled.**

Pipeline:
1. Load pretrained GPT-2 weights from OpenAI
2. Fine-tune on Shakespeare (creative writing)
3. Fine-tune on GSM8K (math word problems)

In [ ]:
!pip install -q transformers datasets tqdm

In [ ]:
import os
import re
import math
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using: cuda
GPU: Tesla T4


---
## 1. GPT-2 Model Architecture

Built from scratch. Matches OpenAI's GPT-2 Small exactly:
- 12 transformer layers
- 12 attention heads
- 768 hidden dimension
- Weight-tied embeddings

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return F.layer_norm(x, (self.weight.shape[0],), self.weight, self.bias, eps=1e-5)


class CausalSelfAttention(nn.Module):
    """Multi-head masked self-attention.
    
    Each token can only attend to previous tokens (causal mask).
    Q, K, V come from a single combined linear layer (GPT-2 style).
    """
    def __init__(self, config):
        super().__init__()
        self.n_head = config["n_head"]
        self.n_embd = config["n_embd"]
        
        self.c_attn = nn.Linear(config["n_embd"], 3 * config["n_embd"])
        self.c_proj = nn.Linear(config["n_embd"], config["n_embd"])
        
        # Causal mask: lower triangle = 1 (allowed), upper = 0 (blocked)
        mask = torch.tril(torch.ones(config["block_size"], config["block_size"]))
        self.register_buffer("bias", mask.view(1, 1, config["block_size"], config["block_size"]))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
    """Feed-forward network with GELU activation."""
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config["n_embd"], 4 * config["n_embd"])
        self.c_proj = nn.Linear(4 * config["n_embd"], config["n_embd"])

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        return x


class Block(nn.Module):
    """Transformer block: attention + MLP with residual connections."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config["n_embd"])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config["n_embd"])
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    """Full GPT-2 model."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.wte = nn.Embedding(config["vocab_size"], config["n_embd"])
        self.wpe = nn.Embedding(config["block_size"], config["n_embd"])
        self.h = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.ln_f = LayerNorm(config["n_embd"])
        self.lm_head = nn.Linear(config["n_embd"], config["vocab_size"], bias=False)
        
        self.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = tok_emb + pos_emb
        
        for block in self.h:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config["block_size"]:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


def load_pretrained_weights(model):
    """Load OpenAI GPT-2 weights into our custom model.
    
    Downloads the real GPT-2 from HuggingFace and copies
    matching parameters by name.
    """
    print("Loading pretrained GPT-2 weights...")
    hf_model = AutoModelForCausalLM.from_pretrained("gpt2")
    hf_sd = hf_model.state_dict()
    
    our_sd = model.state_dict()
    copied = 0
    for hf_name, hf_param in hf_sd.items():
        our_name = hf_name.replace("transformer.", "")
        if our_name in our_sd and hf_param.shape == our_sd[our_name].shape:
            our_sd[our_name] = hf_param
            copied += 1
    
    model.load_state_dict(our_sd, strict=False)
    print(f"Copied {copied} weight tensors from pretrained GPT-2")
    return model


# Config — matches GPT-2 Small (124M)
config = {
    "vocab_size": 50257,
    "block_size": 512,
    "n_layer": 12,
    "n_head": 12,
    "n_embd": 768,
}

# Build model and load weights
model = GPT(config).to(device)
model = load_pretrained_weights(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading pretrained GPT-2 weights...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Copied 112 weight tensors from pretrained GPT-2
Total parameters: 124,046,592


---
## 2. Data Preparation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocabulary size: {tokenizer.vocab_size}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 50257


In [ ]:
print("Downloading Shakespeare:")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
shakespeare_text = response.text
print(f"Shakespeare: {len(shakespeare_text):,} characters")

# Split into chunks
shakespeare_chunks = [c.strip() for c in shakespeare_text.split("\n\n") if len(c.strip()) > 50]
print(f"{len(shakespeare_chunks)} chunks")

Shakespeare: 1,115,394 characters
5174 chunks


In [ ]:
class TextDataset(Dataset):
    """Language modeling dataset: chunks of text tokenized as (input, target) pairs."""
    def __init__(self, texts, tokenizer, block_size):
        all_ids = []
        for text in texts:
            all_ids.extend(tokenizer.encode(text))
        
        self.inputs = []
        for i in range(0, len(all_ids) - block_size, block_size):
            chunk = all_ids[i:i + block_size + 1]
            self.inputs.append((
                torch.tensor(chunk[:-1], dtype=torch.long),
                torch.tensor(chunk[1:], dtype=torch.long),
            ))
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx]

shakespeare_data = TextDataset(shakespeare_chunks, tokenizer, config["block_size"])
print(f"Shakespeare dataset: {len(shakespeare_data):,} samples")

Shakespeare dataset: 582 samples


In [ ]:
print("Loading GSM8K:")
gsm8k = load_dataset("gsm8k", "main", split="train")
print(f"GSM8K: {len(gsm8k):,} training problems")

# subset for training (5000 problems)
gsm8k = gsm8k.select(range(5000))
print(f"Using subset: {len(gsm8k)} problems")

Loading GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K: 7,473 training problems
Using subset: 5000 problems


In [ ]:
class GSM8KDataset(Dataset):
    """GSM8K math problems formatted as Q&A pairs."""
    def __init__(self, dataset, tokenizer, block_size):
        self.inputs = []
        for example in dataset:
            text = f"Question: {example['question']}\nAnswer: {example['answer']}"
            ids = tokenizer.encode(text)
            if len(ids) <= block_size + 1:
                padded = ids + [tokenizer.eos_token_id] * (block_size + 1 - len(ids))
                self.inputs.append((
                    torch.tensor(padded[:block_size], dtype=torch.long),
                    torch.tensor(padded[1:block_size + 1], dtype=torch.long),
                ))
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx]

gsm8k_data = GSM8KDataset(gsm8k, tokenizer, config["block_size"])
print(f"GSM8K dataset: {len(gsm8k_data):,} samples")

GSM8K dataset: 5,000 samples


---
## 3. Training

In [ ]:
def train(model, dataset, tokenizer, num_epochs, batch_size, lr, label):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(loader) * num_epochs
    )
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"{label} epoch {epoch+1}/{num_epochs}")
        for step, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
        avg_loss = epoch_loss / len(loader)
        print(f"{label} epoch {epoch+1}: avg loss = {avg_loss:.4f}")
        
        # Generate sample after each epoch
        generate_sample(model, tokenizer, label)
    
    return model


def generate_sample(model, tokenizer, label, prompt=None, max_new=100):
    """Generate a sample to check training progress."""
    if prompt is None:
        prompt = "\n" if "shake" in label.lower() else "Question:"
    
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=max_new, temperature=0.8)
    
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"\n--- {label} sample ---")
    print(generated[:200])
    print("-" * 30)
    model.train()


def compute_perplexity(model, dataset, num_samples=100):
    model.eval()
    losses = []
    loader = DataLoader(dataset, batch_size=4, shuffle=True, drop_last=False)
    for i, (x, y) in enumerate(loader):
        if i >= num_samples:
            break
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    avg_loss = sum(losses) / len(losses)
    perplexity = math.exp(avg_loss)
    return perplexity, avg_loss

In [ ]:
# Stage 1: Train on Shakespeare
os.makedirs("checkpoints", exist_ok=True)

model = train(
    model, shakespeare_data, tokenizer,
    num_epochs=5,
    batch_size=4,
    lr=5e-5,
    label="Shakespeare",
)

torch.save(model.state_dict(), "checkpoints/shakespeare.pt")
print("Shakespeare checkpoint saved!")

# Evaluate Shakespeare BEFORE Stage 2
shake_ppl_before, shake_loss_before = compute_perplexity(model, shakespeare_data, num_samples=50)
print(f"Shakespeare before Stage 2: PPL={shake_ppl_before:.2f}, Loss={shake_loss_before:.4f}")

Shakespeare epoch 1/5: 100%|██████████| 145/145 [01:24<00:00,  1.72it/s, loss=6.7799]


Shakespeare epoch 1: avg loss = 7.1866

--- Shakespeare sample ---

:

!. to should.,less while
;! make

,:


, with,,
 not behold to toGod! mypire OF of

 him never.COR my!D a the a liar, you whereHis, him all. this and never forth!;,, shame odd;
IIs!,TwIO king, am 
------------------------------


Shakespeare epoch 2/5: 100%|██████████| 145/145 [01:35<00:00,  1.52it/s, loss=6.1449]


Shakespeare epoch 2: avg loss = 6.4395

--- Shakespeare sample ---


I's?
From,
KE along from Caesar,tis,
And you speak-EL:


That her with sixteenant with '
 we

Were away on be with.
Is or:R, and I pr that for the to
What he as's I, kn womanIZ,

This kingKING shall
------------------------------


Shakespeare epoch 3/5: 100%|██████████| 145/145 [01:34<00:00,  1.53it/s, loss=6.1404]


Shakespeare epoch 3: avg loss = 6.1637

--- Shakespeare sample ---

A not
So have know the be a the
Inst you and
For thy
I
F
ThisLE how of,
Think:
No have being of you,:
And all cold but that world sir you.this fit.
To:
Or:O
The thee, strict!
PETKE is noble please,
T
------------------------------


Shakespeare epoch 4/5: 100%|██████████| 145/145 [01:34<00:00,  1.54it/s, loss=6.0575]


Shakespeare epoch 4: avg loss = 6.0347

--- Shakespeare sample ---


Go him than:
And up:Mno nor o I;
Unt, my to, to the my,:ICKESS we times!
The I
And man.
And deathest not myine this, IrosKE:
O
S!
I I fight ears cord I his; how,
So, my!
And, what to, hear, shall th
------------------------------


Shakespeare epoch 5/5: 100%|██████████| 145/145 [01:34<00:00,  1.54it/s, loss=5.9217]


Shakespeare epoch 5: avg loss = 5.9771

--- Shakespeare sample ---


How is:
My love
Here'd me:
Bnews? your him,
Which,
See, in, with war, keeper:
He allegiance,
It she any purpose.
That, a it.
And.
Some, that and;:
Of, CapIC, the andan thy! much flame,
Un not subjec
------------------------------
Shakespeare checkpoint saved!
Shakespeare before Stage 2: PPL=386.39, Loss=5.9569


In [ ]:
# Stage 2: Train on GSM8K + Shakespeare mix
# Mix 10% Shakespeare with GSM8K to prevent catastrophic forgetting
class MixedDataset(Dataset):
    def __init__(self, ds1, ds2, ratio, size=10000):
        self.ds1 = ds1
        self.ds2 = ds2
        self.ratio = ratio
        self.size = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if torch.rand(1).item() < self.ratio:
            return self.ds1[torch.randint(0, len(self.ds1), (1,)).item()]
        else:
            return self.ds2[torch.randint(0, len(self.ds2), (1,)).item()]

mixed_data = MixedDataset(gsm8k_data, shakespeare_data, ratio=0.9, size=10000)

model = train(
    model, mixed_data, tokenizer,
    num_epochs=10,
    batch_size=4,
    lr=3e-5,
    label="GSM8K+Shakespeare",
)

torch.save(model.state_dict(), "checkpoints/tinygpt_shake_gsm.pt")
print("Final checkpoint saved!")

GSM8K+Shakespeare epoch 1/10: 100%|██████████| 2500/2500 [27:07<00:00,  1.54it/s, loss=2.5463]


GSM8K+Shakespeare epoch 1: avg loss = 1.9972

--- GSM8K+Shakespeare sample ---

And is---
I, got Romeo?
And I is the better.WARWAR,, the souls,
I:
Haret'd,
Yet thee?
How triumph, it held'd this news
Is the widow:
What?
Yet unis's will he would be,
I and something gives my the ma
------------------------------


GSM8K+Shakespeare epoch 2/10: 100%|██████████| 2500/2500 [27:10<00:00,  1.53it/s, loss=1.2644]


GSM8K+Shakespeare epoch 2: avg loss = 1.7505

--- GSM8K+Shakespeare sample ---

Which, I you for whom my honour:
Were brother'd he gave my lies to thought away.
Most my command, be them
I'd to dalt than my love
the dire, so let
The soul of seven season.
Yet you he'll I have the 
------------------------------


GSM8K+Shakespeare epoch 3/10: 100%|██████████| 2500/2500 [27:07<00:00,  1.54it/s, loss=1.2600]


...

GSM8K+Shakespeare epoch 8: avg loss = 1.3118

--- GSM8K+Shakespeare sample ---

R and be taken,
To bring her so they be thee up
Here is them down or a fingered
I would fade.CLANUS:
Which from his grace.Burse:
That make an man's season? I not to rest.CLINIUS:
So so I am else I be
------------------------------


GSM8K+Shakespeare epoch 9/10: 100%|██████████| 2500/2500 [27:08<00:00,  1.53it/s, loss=0.7509]


GSM8K+Shakespeare epoch 9: avg loss = 1.2767

--- GSM8K+Shakespeare sample ---

The the car so
For een last this
The alive, disguised, I am a man
And might be nice;
When he had to watch you,
If it was whom I will have a courses,
Will I have to
To the doth where you that cast it

------------------------------


GSM8K+Shakespeare epoch 10/10: 100%|██████████| 2500/2500 [27:12<00:00,  1.53it/s, loss=2.1330]


GSM8K+Shakespeare epoch 10: avg loss = 1.2725

--- GSM8K+Shakespeare sample ---

As virtue too in velvet
To send his own, go to whose to him
there and, my majesty,
I cannot live myointed the man,
And that, but well you, nor my satisfaction, if to the,
With your sword, to my wife,
------------------------------
Final checkpoint saved!


---
## 4. Evaluation

In [ ]:
# Generate Shakespeare samples
prompts = [
    "To be, or not to be",
    "Romeo, Romeo, wherefore art thou",
    "All that glitters is not gold",
]

model.eval()
print("=== Shakespeare Samples ===\n")
for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=150, temperature=0.8)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"\n{generated[:300]}")
    print("\n" + "-" * 40 + "\n")

=== Shakespeare Samples ===

Prompt: To be, or not to be

To be, or not to bequies:
I had me, I have not
A counsel, but love,
You will a earthquake: a times like the wife
A to him
Which they have to my father,
Why than I be not forfeit shall counterfeit. Thyointed you? why.
He have be seen him;
Will his general.Third Senator:
em and every custom my own oth

----------------------------------------

Prompt: Romeo, Romeo, wherefore art thou

Romeo, Romeo, wherefore art thou.
I thou hast the fair this protect thee, and,
Shall weeping and a mildly blood,
See where he gone,
How belips,--Second Rivers as many more than the rancvelans of my recies.
And I bwled the knees.
The quarter of a months away, he woe,
Where did I am not call-day,
I be

----------------------------------------

Prompt: All that glitters is not gold

All that glitters is not gold,
With our word death is at this
I will be done, that:
In the lips,
Can granted in the birth, with the fair gates of my thing which of their
But

In [ ]:
# Evaluate: Perplexity + Format Learning Score
print("=" * 50)
print("FINAL EVALUATION")
print("=" * 50)

# 1. Perplexity on math data
ppl, loss_val = compute_perplexity(model, gsm8k_data, num_samples=50)
print(f"\n[1/4] GSM8K Perplexity: {ppl:.2f}")
print(f"       Cross-Entropy Loss: {loss_val:.4f}")
print(f"       (Lower = model learned the Q&A format better)")

# 2. Generate math samples side-by-side
print(f"\n[2/4] Sample Generations:")
test_data = load_dataset("gsm8k", "main", split="test")
model.eval()
for i in range(5):
    ex = test_data[i]
    prompt = f"Question: {ex['question']}\nAnswer:"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=80, temperature=0.3)
    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    pred = pred[len(prompt):].strip()
    expected = ex["answer"].strip()
    print(f"\nQ: {ex['question'][:80]}...")
    print(f"Expected: {expected[:80]}...")
    print(f"Model:   {pred[:80]}...")
    print("-" * 40)

# 3. Contains-answer accuracy
print(f"\n[3/4] Answer-Contains Accuracy (200 problems):")
def extract_answer(text):
    match = re.search(r'####\s*(-?\d+\.?\d*)', text)
    return match.group(1) if match else None

test_subset = test_data.select(range(200))
correct_exact = 0
correct_contains = 0
total = 0

for ex in test_subset:
    prompt = f"Question: {ex['question']}\nAnswer:"
    expected_answer = extract_answer(ex["answer"])
    if expected_answer is None:
        continue
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=100, temperature=0.3)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predicted_answer = extract_answer(generated)
    
    if predicted_answer == expected_answer:
        correct_exact += 1
    if predicted_answer and expected_answer in generated:
        correct_contains += 1
    total += 1

exact_acc = correct_exact / total * 100
contains_acc = correct_contains / total * 100
print(f"  Exact answer match:     {correct_exact}/{total} = {exact_acc:.1f}%")
print(f"  Answer in generated text: {correct_contains}/{total} = {contains_acc:.1f}%")

# 4. Perplexity on Shakespeare
ppl_shake, loss_shake = compute_perplexity(model, shakespeare_data, num_samples=50)
print(f"\n[4/4] Shakespeare Perplexity:")
print(f"       BEFORE Stage 2: PPL={shake_ppl_before:.2f}, Loss={shake_loss_before:.4f}")
print(f"       AFTER  Stage 2: PPL={ppl_shake:.2f}, Loss={loss_shake:.4f}")
print(f"       (Before = pure Shakespeare; After = GSM8K + 10% Shakespeare mix)")
if ppl_shake < shake_ppl_before:
    print(f"       -> Shakespeare quality IMPROVED (mix helped!)")
else:
    print(f"       -> Shakespeare degraded by {ppl_shake - shake_ppl_before:.2f} PPL")

print(f"\n{'=' * 50}")
print(f"SUMMARY: Perplexity={ppl:.2f} (GSM8K), {ppl_shake:.2f} (Shakespeare, was {shake_ppl_before:.2f})")
print(f"         Contains-answer rate: {contains_acc:.1f}%")
print(f"{'=' * 50}")

FINAL EVALUATION

[1/4] GSM8K Perplexity: 2.48
       Cross-Entropy Loss: 0.9094
       (Lower = model learned the Q&A format better)

[2/4] Sample Generations:

Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
Expected: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2...
Model:   The total of the cost of the number of the morning is $3.50 * $2 = $<<2*2=2>>2.
...
----------------------------------------

Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bol...
Expected: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1...
Model:   The rectangular of the number of the rectangle is 2*2=<<2*2=1200>>1200 meters
So...
----------------------------------------

Q: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts...
Expected: The cost of the house and repairs came out to 80,000+50,000=$<<80000+50000=13000...
Model:   He needs to pay 